In [1]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/09 09:54:36 WARN Utils: Your hostname, Dinadis-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.13.45.161 instead (on interface en0)
26/10/09 09:54:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/09 09:54:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import pandas as pd
CURATED_DIR = "../data/curated/curated_transactions/part_3.parquet"

# Load the ETL curated transaction level dataset 
curated_df = pd.read_parquet(CURATED_DIR)
print(curated_df.columns.tolist())
print(curated_df.info())

['user_id', 'merchant_abn', 'dollar_value', 'order_id', 'order_datetime', 'name_consumer', 'address', 'state', 'postcode', 'gender', 'consumer_id', 'name_merchant', 'tags', 'consumer_fraud_probability', 'merchant_fraud_probability', 'consumer_is_fraud', 'merchant_is_fraud', 'POA_CODE_2021', 'SA2_CODE_2021', 'mb_count', 'poa_total', 'ratio', 'SA2_NAME_2021', 'disadvantage_score', 'disadvantage_decile', 'adv_disadv_score', 'adv_disadv_decile', 'economic_resources_score', 'economic_resources_decile', 'education_occupation_score', 'education_occupation_decile', 'usual_resident_population', 'SA2_NAME_2021_income', 'IRSAD_score', 'IRSAD_decile', 'median_personal_income_weekly', 'median_family_income_weekly', 'median_household_income_weekly', 'disadvantage_score_is_approximated', 'disadvantage_decile_is_approximated', 'adv_disadv_score_is_approximated', 'adv_disadv_decile_is_approximated', 'economic_resources_score_is_approximated', 'economic_resources_decile_is_approximated', 'education_occu

In [6]:
FEATURE_DIR = "../data/merchant_features.parquet"

# Load the ETL curated transaction level dataset 
feat_df = pd.read_parquet(FEATURE_DIR)
print(feat_df.columns.tolist())
print(feat_df.info())


['merchant_abn', 'total_revenue', 'n_transactions', 'avg_order_value', 'n_customers', 'orders_per_customer', 'avg_weekly_transactions', 'repeat_rate', 'top_customer_share', 'revenue_growth', 'revenue_cov', 'active_months', 'name_merchant', 'category', 'revenue_level', 'take_rate', 'has_merchant_info', 'consumer_fraud_rate', 'fraud_value_share', 'merchant_fraud_rate', 'avg_consumer_fraud_prob', 'max_merchant_fraud_prob', 'n_postcodes', 'top_postcode_share', 'top5_postcode_share', 'no_sa2_match', 'w_adv_disadv_decile', 'w_economic_resources_decile', 'expected_bnpl_revenue', 'flag_small_base_big_spender', 'flag_large_base_high_fraud', 'flag_new_merchant']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4379 entries, 0 to 4378
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   merchant_abn                 4379 non-null   int64  
 1   total_revenue                4379 non-null   float64


In [ ]:
cur_with_feat = curated_df.merge(
    feat_df, 
    on="merchant_abn"  
)
print(cur_with_feat.info)
print(cur_with_feat.shape)

<bound method DataFrame.info of          user_id  merchant_abn  dollar_value  \
0              1   45629217853     43.738115   
1          14936   24852446429      9.788472   
2              3   32656335530    100.160247   
3          14936   72472909171     28.540345   
4              4   86223953880    313.021668   
...          ...           ...           ...   
4508101    14931   94455880010     93.759048   
4508102    14931   35755909141    124.307113   
4508103    14931   79417999332    131.169882   
4508104    14932   46804135891     37.889414   
4508105    14932   35223308778    196.988323   

                                     order_id order_datetime  \
0        809b9b93-0a7d-4627-abd8-b6899d13f169     2021-08-28   
1        5a04b94c-c553-47c6-b99b-a9697b977984     2021-08-28   
2        8776872a-a340-46ff-84c8-a024453ff8c3     2021-08-28   
3        e997036a-1418-44c6-8629-6691a2975297     2021-08-28   
4        6161bda6-030c-42c0-8bd6-9067f18d4223     2021-08-28   
...    

,user_id,merchant_abn,dollar_value,order_id,order_datetime,name_consumer,address,state,postcode,gender,...,n_postcodes,top_postcode_share,top5_postcode_share,no_sa2_match,w_adv_disadv_decile,w_economic_resources_decile,expected_bnpl_revenue,flag_small_base_big_spender,flag_large_base_high_fraud,flag_new_merchant
0,1,45629217853,43.738115,809b9b93-0a7d-4627-abd8-b6899d13f169,2021-08-28,Yolanda Williams,413 Haney Gardens Apt. 742,WA,6935,Female,...,3164,0.001141,0.004891,0.163557,4.901554,5.312844,187186.419373,False,True,False
1,14936,24852446429,9.788472,5a04b94c-c553-47c6-b99b-a9697b977984,2021-08-28,Maria Riley,416 Phillip View Suite 130,SA,5157,Female,...,3163,0.000908,0.004352,0.166016,4.901815,5.307321,81454.753831,False,False,False
2,3,32656335530,100.160247,8776872a-a340-46ff-84c8-a024453ff8c3,2021-08-28,Jill Jones MD,40693 Henry Greens,NT,0862,Female,...,2386,0.002274,0.010574,0.163965,4.900107,5.270114,10811.353120,False,False,False
3,14936,72472909171,28.540345,e997036a-1418-44c6-8629-6691a2975297,2021-08-28,Maria Riley,416 Phillip View Suite 130,SA,5157,Female,...,3154,0.001519,0.005595,0.166049,4.898206,5.305680,143879.159171,False,False,False
4,4,86223953880,313.021668,6161bda6-030c-42c0-8bd6-9067f18d4223,2021-08-28,Lindsay Jimenez,00653 Davenport Crossroad,NSW,2780,Female,...,1153,0.004189,0.018200,0.162143,4.787970,5.230035,7944.184393,False,False,False
5,14938,53546629749,659.435943,23519a0e-db53-4b82-9726-bbc35fbec0f9,2021-08-28,Jasmine Ford,1309 Karen Extension,NSW,2625,Undisclosed,...,168,0.019605,0.089777,0.137143,5.215681,5.550863,1544.952210,False,False,False
6,7,17488304283,67.954953,e637ca66-ed07-42c3-a39b-d49a1f97dde8,2021-08-28,Andrea Jones,122 Brandon Cliff,QLD,4606,Female,...,2976,0.001495,0.006854,0.165769,4.914588,5.293796,65131.543646,False,False,False
7,14938,98269572896,86.809978,a79e1147-9922-4d92-a79f-ca5c43bfc5d6,2021-08-28,Jasmine Ford,1309 Karen Extension,NSW,2625,Undisclosed,...,3058,0.001585,0.006968,0.167595,4.916253,5.348737,44490.371866,False,False,False
8,7,71350572766,113.726665,d6743e55-2463-49c9-8670-124594295271,2021-08-28,Andrea Jones,122 Brandon Cliff,QLD,4606,Female,...,2378,0.002544,0.010839,0.168551,4.982896,5.335121,5981.601530,False,False,False
9,14940,54705111450,100.873253,a44bef39-26db-4da3-ab9f-16f1d231ba43,2021-08-28,Theresa Maldonado,5004 Cordova Vista,NSW,2333,Female,...,1507,0.004777,0.019327,0.166905,4.870398,5.290663,6979.175263,False,False,False
